In [ ]:
# Figure 3 and Figure A.9
# Non-uniform twin search results for increasing number of hyperedges m

In [ ]:
import math

import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
label_size = 20
tick_size = 18
legend_font = 18
labelpad = 3
mpl.rcParams["axes.labelpad"] = labelpad
mpl.rcParams["axes.labelsize"] = label_size
mpl.rcParams["legend.fontsize"] = legend_font
mpl.rcParams["xtick.labelsize"] = tick_size
mpl.rcParams["ytick.labelsize"] = tick_size
text_label_x = 0.075
text_label_y = 0.975

In [ ]:
import sys
sys.path.append("../python/")
from distance_stats import read_data
from projection_utils import *

In [ ]:
def read_exact_mates_pairs_file(filename):
    exact_mates = dict()
    with open(filename) as fin:
        for proj_idx, line in enumerate(fin):
            exact_mates[proj_idx] = []
            # Split on |
            exact_str = line.strip().split(";")
            if len(exact_str) > 0:
                for pair_str in exact_str:
                    exact_mates[proj_idx].append(tuple(pair_str.strip().split(",")))
    return exact_mates

In [ ]:
def get_max_width_dist(output_data, n):
    max_widths = []
    # Compute exact max widths rather than using the C++ estimate
    for proj_idx in range(len(output_data["hypergraphs"])):
        # Find a 3-uniform hypergraph and compute its projection
        unif = False
        unif_h = None
        for h in output_data["hypergraphs"][proj_idx]:
            if all([len(he) == 3 for he in h]):
                unif_h = h
                unif = True
                break
                
        if not unif:
            print("Never found a 3-uniform hypergraph...")

        proj = get_projection(n, unif_h)
        emw = get_exact_max_width(proj)
        max_widths.append(emw)
    return max_widths

In [ ]:
k = 3
n = 6
results_dir = "../../results/reproduce/exhaustive/"
m_vals = list(range(2, 21))

In [ ]:
max_width_means = []
max_width_bps = []

num_cliques_bps = []

twins_bps = []

iso_classes_bps = []

runtime_bps = []

size_dist_bps = []
exact_pairs_counts = dict()
iso_pairs_counts = dict()
for m in m_vals:
    print(f"m: {m}")
    # Read the data
    filename = f"n-{n}_m-{m}_k-{k}_non-uniform_exhaustive_projections.csv"
    output_data = read_data(results_dir + filename, xgi_hypergraphs=False)
    
    # Get the average exponent
    max_width_means.append(np.mean(output_data["max_width"]))
    max_width_bps.append(list(map(math.log10, get_max_width_dist(output_data, n))))

    num_cliques_bps.append(output_data["num_cliques"])
    
    twins_bps.append(output_data["total_twins"])
    
    iso_classes_bps.append(output_data["total_filtered"])
    
    # Get the average runtime
    runtime_bps.append(output_data["runtime"])
    
    size_dist_bps.append([len(d) for d in output_data["dist_results_dict"]])
    
    # Gram mates
    iso_pairs_counts[m] = np.array([v for v in output_data["total_mates"] if v > 0])
    exact_pairs_counts[m] = np.array([len(pairs) for pairs in read_exact_mates_pairs_file(results_dir + filename + ".exact_mates_pairs").values() if len(pairs) > 0])
    del output_data

# Figure 3

In [ ]:
fig, axs = plt.subplots(3, 2, squeeze=False, figsize=(10, 9), constrained_layout=True)
#fig.subplots_adjust(wspace=0.3, hspace=0.25)
row = 0
col = 0

xt = m_vals[::2]

# m VERSUS TWINS
axs[row][col].boxplot(twins_bps, positions=m_vals, whis=(0,100))
axs[row][col].set_xticks(xt, xt)
#axs[row][col].set_xlabel("$m$")
axs[row][col].set_ylabel("Non-uniform Twins")
axs[row][col].set_yscale("log")
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].text(text_label_x, text_label_y, "(a)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)


col = 1
axs[row][col].boxplot(iso_classes_bps, positions=m_vals, whis=(0,100))
axs[row][col].set_xticks(xt, xt)
#axs[row][col].set_xlabel("$m$")
axs[row][col].set_ylabel("Iso. Classes")
axs[row][col].set_yscale("log")
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].text(text_label_x, text_label_y, "(b)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)

row = 1
col = 0
# m VERSUS size dists
axs[row][col].boxplot(size_dist_bps, positions=m_vals, whis=(0,100))
axs[row][col].set_xticks(xt, xt)
axs[row][col].set_ylabel("Size Dists.")
axs[row][col].set_yscale("log")
#ymin = np.log10(runtime_bps)
#axs[row][col].set_ylim(10**np.log10(runtime_bps).min(), 10**10)
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].text(text_label_x, text_label_y, "(c)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)

col = 1
# m VERSUS CLIQUES
axs[row][col].boxplot(num_cliques_bps, positions=m_vals, whis=(0,100))
axs[row][col].set_xticks(xt, xt)
axs[row][col].set_ylim(0, 50)
axs[row][col].set_ylabel("Number of Cliques")
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].text(text_label_x, text_label_y, "(d)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)


row = 2

col = 0
# m VERSUS WORST CASE
axs[row][col].boxplot(max_width_bps, positions=m_vals, whis=(0,100))
axs[row][col].set_xticks(xt, xt)
# Ticks must come from what is plotted. max_width_bps holds the exact
# widths recomputed in Python; max_width_means holds the C++ estimate,
# which is a different quantity and differs by 1-2 orders of magnitude.
_plotted = [v for bp in max_width_bps for v in bp]
yt = list(range(int(min(_plotted)), int(max(_plotted))+10, 10))
axs[row][col].set_yticks(yt, map(lambda v : f"$10^{{{v}}}$", yt))
axs[row][col].set_ylabel("Worst Case Tree Size")
axs[row][col].set_xlabel("$m$")
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].text(text_label_x, text_label_y, "(e)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)


col = 1
# m VERSUS RUNTIME
axs[row][col].boxplot(runtime_bps, positions=m_vals, whis=(0,100))
axs[row][col].set_xticks(xt, xt)
axs[row][col].set_ylabel("Runtime (ms)")
axs[row][col].set_ylim(10**-1, 10**10)
axs[row][col].set_yscale("log")
axs[row][col].set_xlabel("$m$")
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].text(text_label_x, text_label_y, "(f)", size=label_size, fontweight="bold", horizontalalignment='center',
     verticalalignment='center', transform=axs[row][col].transAxes)

fig.savefig("../plots/non-uniform-stats.pdf", bbox_inches="tight", dpi=450)

# Figure A.9

In [ ]:
fig, axs = plt.subplots(1, 2, squeeze=False, figsize=(10, 3), constrained_layout=True)
#fig.subplots_adjust(wspace=0.3, hspace=0.25)
row = 0
col = 0
x_exact = [k for k in exact_pairs_counts.keys() if len(exact_pairs_counts[k]) > 0]
axs[row][col].boxplot([v for v in exact_pairs_counts.values() if len(v) > 0], positions=x_exact, showfliers=False, whis=(0,100))
axs[row][col].plot(x_exact, [len(val) for val in exact_pairs_counts.values() if len(val) > 0], label="# Proj. w/ Mates")
axs[row][col].set_yscale("log")
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].set_ylabel("Exact Mates")
axs[row][col].set_xlim(min(xt)-1, max(xt)+1)
axs[row][col].set_xticks(xt, xt)
axs[row][col].set_xlabel("$m$")
axs[row][col].text(text_label_x, text_label_y, "(a)", size=label_size,
                   fontweight="bold", horizontalalignment='center',
                   verticalalignment='center', transform=axs[row][col].transAxes)
axs[row][col].set_ylim(10**-0.1, 10**10)
axs[row][col].legend(frameon=False)

col = 1
x_iso = [k for k in iso_pairs_counts.keys() if len(iso_pairs_counts[k]) > 0]
axs[row][col].boxplot([v for v in iso_pairs_counts.values() if len(v) > 0], positions=x_iso, showfliers=False, whis=(0,100))
axs[row][col].plot(x_iso, [len(val) for val in iso_pairs_counts.values() if len(val) > 0])
axs[row][col].set_yscale("log")
axs[row][col].spines[['right', 'top']].set_visible(False)
axs[row][col].set_ylabel("Iso. Mates")
axs[row][col].set_xlim(min(xt)-1, max(xt)+1)
axs[row][col].set_xticks(xt, xt)
axs[row][col].set_xlabel("$m$")
axs[row][col].set_ylim(10**-0.1, 10**10)
axs[row][col].text(text_label_x, text_label_y, "(b)", size=label_size,
                   fontweight="bold", horizontalalignment='center',
                   verticalalignment='center', transform=axs[row][col].transAxes)

fig.savefig("../plots/gram-mates-non-uniform-stats.pdf", bbox_inches="tight", dpi=450)